In [ ]:
!pip install unsloth
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy
!pip install datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detect (Float16 for T4)
load_in_4bit = True # Essential for free Colab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_dataset

prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
Convert the following natural language question into a SQL query based on the provided database schema.

### Input:
Schema: {}
Question: {}

### Response:
{}"""

dataset = load_dataset("xlangai/spider", split = "train")

def formatting_prompts_func(examples):
    schemas = examples["question"] # Spider doesn't have a direct 'schema' string, you'd usually extract this from the .sqlite files.
    # For a quick viva-ready demo, we'll focus on the 'question' and 'query' mapping.
    questions = examples["question"]
    queries   = examples["query"]
    texts = []
    for question, query in zip(questions, queries):
        # In a real scenario, you'd fetch the specific schema for the DB ID here
        text = prompt_style.format("Refer to database context.", question, query) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase to 300+ for better accuracy if you have time
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.754233
2,2.580664
3,2.702683
4,2.282264
5,2.039440
6,1.940805
7,1.442753
8,1.298633
9,0.923654
10,0.823392


TrainOutput(global_step=60, training_loss=0.7880913943052292, metrics={'train_runtime': 216.0051, 'train_samples_per_second': 2.222, 'train_steps_per_second': 0.278, 'total_flos': 2693326149058560.0, 'train_loss': 0.7880913943052292, 'epoch': 0.06857142857142857})

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    prompt_style.format(
        "Table: Students, Columns: [id, name, age, grade]",
        "Show me all students who are older than 20.",
        "" # Leave response blank for generation
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64)
tokenizer.batch_decode(outputs)

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request.\n\n### Instruction:\nConvert the following natural language question into a SQL query based on the provided database schema.\n\n### Input:\nSchema: Table: Students, Columns: [id, name, age, grade]\nQuestion: Show me all students who are older than 20.\n\n### Response:\nSELECT * FROM Students WHERE age > 20<|end_of_text|>']

In [ ]:
# # Save the model in GGUF format (4-bit quantization for speed)
# model.save_pretrained_gguf("sql_model", tokenizer, quantization_method = "q4_k_m")

# # Download the file to your computer
# from google.colab import files
# files.download("sql_model/unsloth.Q4_K_M.gguf")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 2/4 [00:43<00:43, 21.86s/it]


KeyboardInterrupt: 

In [ ]:
# This will take 3-5 minutes because it has to 'convert' the safetensors to GGUF
model.save_pretrained_gguf(
    "sql_model",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:20<00:00, 110.19s/it]


Unsloth: Merge process complete. Saved to `/content/sql_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['sql_model_gguf/llama-3-8b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['sql_model_gguf/llama-3-8b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/llama-3-8b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model sql_model_gguf/llama-3-8b.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'sql_model',
 'gguf_directory': 'sql_model_gguf',
 'gguf_files': ['sql_model_gguf/llama-3-8b.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
import os
import glob
import shutil
from google.colab import drive

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Search for the GGUF file anywhere in the 'sql_model' folder
# This handles cases where the path might be slightly different
search_pattern = "./*.gguf"
found_files = glob.glob(search_pattern)

if len(found_files) > 0:
    source_path = found_files[0] # Take the first GGUF file found
    target_folder = '/content/drive/MyDrive/SQL_Model_Project'

    os.makedirs(target_folder, exist_ok=True)
    dest_path = os.path.join(target_folder, os.path.basename(source_path))

    print(f"Found model at: {source_path}")
    print("Transferring to Google Drive... please wait.")

    shutil.copy(source_path, dest_path)
    print(f"✅ Success! Your model is now at: {dest_path}")
else:
    print("❌ Error: No GGUF file found. Did you run the 'save_pretrained_gguf' cell first?")
    # Let's list files to see what's actually there
    print("Current files in sql_model directory:")
    if os.path.exists("sql_model"):
        print(os.listdir("sql_model"))
    else:
        print("The 'sql_model' directory does not even exist!")

❌ Error: No GGUF file found. Did you run the 'save_pretrained_gguf' cell first?
Current files in sql_model directory:
['tokenizer.json', 'model.safetensors.index.json', '.cache', 'config.json', 'model-00004-of-00004.safetensors', 'model-00001-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'tokenizer_config.json']
